In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import time

In [2]:
# ── SOURCE 1: martj42 — goals, results, shootouts (free, one line) ──────────
r = requests.get("https://raw.githubusercontent.com/martj42/international_results/master/results.csv")
results = pd.read_csv(StringIO(r.text))
results['date'] = pd.to_datetime(results['date'])

# Filter to recent matches only — team styles change, old data adds noise
results = results[results['date'] >= '2024-01-01'].copy()

# Filter to competitive matches only (better signal than friendlies for WC)
COMPETITIVE = ['FIFA World Cup', 'FIFA World Cup qualification',
               'UEFA Euro', 'UEFA Euro qualification',
               'Copa América', 'Africa Cup of Nations',
               'AFC Asian Cup', 'CONCACAF Gold Cup',
               'UEFA Nations League', 'CONMEBOL–UEFA Cup of Champions']
results['is_competitive'] = results['tournament'].isin(COMPETITIVE)

# ── SOURCE 2: shootouts ──────────────────────────────────────────────────────
r2 = requests.get("https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv")
shootouts = pd.read_csv(StringIO(r2.text))
shootouts['date'] = pd.to_datetime(shootouts['date'])
# Columns: date, home_team, away_team, winner

# Merge shootout flag onto results
results = results.merge(
    shootouts[['date','home_team','away_team']].assign(went_to_penalties=True),
    on=['date','home_team','away_team'], how='left'
)
results['went_to_penalties'] = results['went_to_penalties'].fillna(False)

/var/folders/bz/yjywszk15kxc8d5vv73k9xcc0000gn/T/ipykernel_16106/2063471236.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results['went_to_penalties'] = results['went_to_penalties'].fillna(False)


In [3]:
print(f"Matches: {len(results)}, with penalties: {results['went_to_penalties'].sum()}")
print(results.head(3))
results.to_csv("data/international_results.csv", index=False)

Matches: 2530, with penalties: 53
        date  home_team  away_team  home_score  away_score tournament  \
0 2024-01-01      Japan   Thailand         5.0         0.0   Friendly   
1 2024-01-01   China PR  Hong Kong         1.0         2.0   Friendly   
2 2024-01-02  Indonesia      Libya         0.0         4.0   Friendly   

        city               country  neutral  is_competitive  went_to_penalties  
0      Tokyo                 Japan    False           False              False  
1  Abu Dhabi  United Arab Emirates     True           False              False  
2    Antalya                Turkey     True           False              False  


In [ ]:
# # All 48 WC 2026 teams with their FBRef squad IDs
# WC_TEAMS = {
#     "Argentina":     "8efac4a9",  "France":       "e3331592",
#     "Spain":         "6b04671b",  "England":      "c4dbe3f5",
#     "Brazil":        "304635c3",  "Portugal":     "bdbf0f78",
#     "Netherlands":   "d2374cfd",  "Germany":      "adff0e00",
#     "Mexico":        "b009a548",  "USA":          "a0b99e50",
#     "Canada":        "30b8b7c7",  "Morocco":      "56bc47f0",
#     "Senegal":       "6b5a6aa7",  "Nigeria":      "77fc9c5a",
#     "Japan":         "48ce7cf5",  "South Korea":  "b7f21eb3",
#     "Australia":     "1d3a9556",  "Iran":         "0d7e7dc3",
#     "Croatia":       "2b4a8ec5",  "Belgium":      "a24a7183",
#     "Switzerland":   "2aa9ec5d",  "Denmark":      "d8ba3c35",
#     "Uruguay":       "870e020f",  "Colombia":     "6748048a",
#     "Ecuador":       "22e4c575",  "Panama":       "feefd9f9",
#     "Costa Rica":    "31ab7d36",  "Jamaica":      "2e58d5e9",
#     "Saudi Arabia":  "c5ecf8b5",  "Qatar":        "4d3a5b2c",
#     "Egypt":         "9e88dc9e",  "Ivory Coast":  "e78e264b",
#     "Algeria":       "0c3c30b9",  "Cameroon":     "73a0b6a3",
#     "Mali":          "0a0ead6d",  "South Africa": "c95ef45d",
#     "New Zealand":   "8ab4d8cd",  "Serbia":       "7df5c040",
#     "Poland":        "9d2a1d55",  "Turkey":       "55a10c36",
#     "Austria":       "f9b26576",  "Ukraine":      "009e8d45",
#     "Slovenia":      "7a1e1d2f",  "Slovakia":     "e5a2e28e",
#     "Norway":        "b1b8e3b4",  "Scotland":     "3ca9afcf",
#     "Portugal":      "bdbf0f78",  "Wales":        "b7c86a4b",
# }

# YEARS = [2024, 2025, 2026]

# all_cards = []
# for team, squad_id in WC_TEAMS.items():
#     for year in YEARS:
#         url = (f"https://fbref.com/en/squads/{squad_id}/{year}/"
#                f"matchlogs/all_comps/misc/{team.replace(' ','-')}"
#                f"-Men-Match-Logs-All-Competitions")
#         try:
#             tables = pd.read_html(url, header=[0,1])
#             df = tables[0]
#             # Flatten multi-level columns
#             df.columns = ['_'.join(str(c) for c in col).strip('_') 
#                          for col in df.columns]
#             df['team'] = team
#             df['year'] = year
#             all_cards.append(df)
#             print(f"  OK: {team} {year}")
#         except Exception as e:
#             print(f"  SKIP: {team} {year} — {e}")
#         time.sleep(4.5)  # Critical — FBRef blocks if you go faster than this

# cards_df = pd.concat(all_cards, ignore_index=True)
# cards_df.to_csv("data/fbref_misc_logs.csv", index=False)

In [4]:
# ── 1. DOWNLOAD ──────────────────────────────────────────────────────────────
print("Downloading results...")
BASE = "https://raw.githubusercontent.com/martj42/international_results/master"
 
results   = pd.read_csv(StringIO(requests.get(f"{BASE}/results.csv").text))
shootouts = pd.read_csv(StringIO(requests.get(f"{BASE}/shootouts.csv").text))
 
results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])
 
print(f"  Results: {len(results):,} rows | Shootouts: {len(shootouts):,} rows")
 
# ── 2. FILTER: 2018 onwards, meaningful competitions ─────────────────────────
# 2018 gives ~6 years of data — enough rolling history, not so much the
# game has changed beyond recognition.
results = results[results['date'] >= '2024-01-01'].copy()
print(f"  After 2024 filter: {len(results):,} matches")
 
COMPETITIVE = {
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa América',
    'Africa Cup of Nations',
    'AFC Asian Cup',
    'AFC Asian Cup qualification',
    'CONCACAF Gold Cup',
    'CONCACAF Nations League',
    'UEFA Nations League',
    'CONMEBOL–UEFA Cup of Champions',
    'Africa Cup of Nations qualification',
}
results['is_competitive'] = results['tournament'].isin(COMPETITIVE)
 
# ── 3. WINNER COLUMNS ────────────────────────────────────────────────────────
# home_or_away_win : 'home' | 'away' | 'draw'  (useful as model target)
# winner           : actual team name            (useful for bracket simulation)
# goal_diff        : home_score - away_score     (signed, positive = home win)
# total_goals      : home_score + away_score
 
results['home_or_away_win'] = np.where(
    results['home_score'] > results['away_score'], 'home',
    np.where(results['home_score'] < results['away_score'], 'away', 'draw')
)
 
results['winner'] = np.where(
    results['home_score'] > results['away_score'], results['home_team'],
    np.where(results['home_score'] < results['away_score'], results['away_team'], 'Draw')
)
 
results['goal_diff']   = results['home_score'] - results['away_score']
results['total_goals'] = results['home_score'] + results['away_score']
 
# ── 4. MERGE SHOOTOUT DATA ───────────────────────────────────────────────────
# shootouts.csv columns: date, home_team, away_team, winner
# 'winner' here is the team that won the shootout (not the 90min result)
 
shootouts_slim = shootouts[['date', 'home_team', 'away_team', 'winner']].rename(
    columns={'winner': 'shootout_winner'}
)
 
results = results.merge(
    shootouts_slim,
    on=['date', 'home_team', 'away_team'],
    how='left'
)
results['went_to_penalties'] = results['shootout_winner'].notna()
 
# For penalty matches, override winner with the shootout winner
# (the 90min score will be a draw — this is the true match winner)
results['winner'] = np.where(
    results['went_to_penalties'],
    results['shootout_winner'],
    results['winner']
)
 
print(f"  Matches that went to penalties: {results['went_to_penalties'].sum()}")
 
# ── 5. CLEAN SHOOTOUTS TABLE ─────────────────────────────────────────────────
# Add loser column and home/away indicator to shootouts
shootouts['loser'] = np.where(
    shootouts['winner'] == shootouts['home_team'],
    shootouts['away_team'],
    shootouts['home_team']
)
shootouts['winner_side'] = np.where(
    shootouts['winner'] == shootouts['home_team'], 'home', 'away'
)
shootouts = shootouts[shootouts['date'] >= '2024-01-01'].copy()
print(f"  Shootouts since 2024: {len(shootouts)}")
 
# ── 6. DERIVED FEATURES USEFUL FOR MODELLING ─────────────────────────────────
# These are match-level aggregates — feed directly into rolling calculations later
 
results['home_win']  = (results['home_or_away_win'] == 'home').astype(int)
results['away_win']  = (results['home_or_away_win'] == 'away').astype(int)
results['is_draw']   = (results['home_or_away_win'] == 'draw').astype(int)
results['home_clean_sheet'] = (results['away_score'] == 0).astype(int)
results['away_clean_sheet'] = (results['home_score'] == 0).astype(int)
results['over_2_5']  = (results['total_goals'] > 2).astype(int)
results['btts']      = (  # both teams to score
    (results['home_score'] > 0) & (results['away_score'] > 0)
).astype(int)
 
# ── 7. SAVE ───────────────────────────────────────────────────────────────────
col_order = [
    'date', 'tournament', 'is_competitive', 'city', 'country', 'neutral',
    'home_team', 'away_team',
    'home_score', 'away_score', 'goal_diff', 'total_goals',
    'home_or_away_win', 'winner',
    'went_to_penalties', 'shootout_winner',
    'home_win', 'away_win', 'is_draw',
    'home_clean_sheet', 'away_clean_sheet',
    'over_2_5', 'btts',
]
results[col_order].to_csv("data/results_clean.csv", index=False)
 
shootouts[['date','home_team','away_team','winner','loser','winner_side']].to_csv(
    "data/shootouts_clean.csv", index=False
)
 
# ── 8. SANITY CHECK ───────────────────────────────────────────────────────────
# df = pd.read_csv("data/results_clean.csv")
# print("\n── results_clean.csv ──")
# print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
# print(df.dtypes.to_string())
# print("\nSample rows:")
# print(df[df['went_to_penalties']].head(3)[
#     ['date','home_team','away_team','home_score','away_score',
#      'winner','went_to_penalties','shootout_winner']
# ].to_string())
 
# print("\nWin distribution:")
# print(df['home_or_away_win'].value_counts())
 
# print("\n── shootouts_clean.csv ──")
# s = pd.read_csv("data/shootouts_clean.csv")
# print(s.head(5).to_string())
# print(f"\nShootout winner_side distribution:\n{s['winner_side'].value_counts().to_string()}")

  Results: 49,390 rows | Shootouts: 677 rows
  After 2024 filter: 2,530 matches
  Matches that went to penalties: 53
  Shootouts since 2024: 53
